# Retrieving images with IIIF

The [International Image Interoperability Framework](https://iiif.io/) (IIIF) is a standard for sharing high-resolution images on the web. It is used by most cultural heritage institutions to provide access to their collections via a IIIF API (Application Programming Interface). Understanding how IIIF works makes it possible to automate the downloading of documents and, in some cases, even retrieve information such as annotations linked to the images.

The result of a IIIF request takes the form of a relatively complex JSON file. Below are several examples of IIIF manifests that you can explore.

Risāla fī Ikhtilāfāt wuqūʿ al-khuṭūṭ fī l-ʾashkāl from the Sipahsalar Library whose manifesto was retrieved via the Aikon VHS platform.
```
https://eida.obspm.fr/eida/iiif/wit404_pdf553/manifest.json
```
France, Tours, Bibliothèque diocésaine - Diocèse 08 whose manifesto was retrieved via the IRHT’s ARCA library.

```
https://api.irht.cnrs.fr/ark:/63955/fufe90re0j42/manifest.json
```

A IIIF manifest is a structured document that describes a digital heritage object by combining descriptive metadata with an ordered list of the images that compose it (including their identifiers, dimensions, and cropping areas).

# Interacting with a IIIF manifest

In [ ]:
import requests

In [ ]:
url = "https://eida.obspm.fr/eida/iiif/wit404_pdf553/manifest.json"

The most common responses are:

- 200: request successful;
- 400: access denied;
- 500: server errors;

[A resource for all the errors a server might return to you](https://http.cat/)

In [ ]:
resultat = requests.get(url)
resultat

In [ ]:
# If the API response is 403, we must then make the endpoint believe that we are a "human" or non-automated user

headers = {
'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
'Accept': 'application/json'
}

resultat = requests.get(url, headers=headers)
resultat

In [ ]:
data = resultat.json()
data

In [ ]:
listeDesUrlDesImages = []

# Here we are navigating within the manifest; the images are stored under sequences > canvases > images > resources > @id
for canvas in data.get('sequences')[0].get('canvases'):
  images = canvas.get('images')
  for image in images:
    resource = image.get('resource')
    if resource.get('@id'):
      url = resource.get('@id')
      listeDesUrlDesImages.append(url)

listeDesUrlDesImages

# Specifying a IIIF Image URI

The request URI for a IIIF image is always structured in the same way:
```
{scheme}://{server}{/prefix}/{identifier}/{region}/{size}/{rotation}/{quality}.{format}
```
```
https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0001.jpg/full/full/0/default.jpg
```

It is therefore possible to pass arguments directly via the IIIF request.

Here, a square from the center of the image:
```
https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0001.jpg/square/max/0/default.jpg
```

Here, an inverted (upside-down) square:
```
https://vhs.huma-num.fr/iiif/2/wit2320_img2562_0001.jpg/square/max/180/default.jpg
```


# Rapatrier des images IIIF

In [ ]:
import time
import os

from tqdm import tqdm

os.makedirs('/content/imagesTéléchargés') # We are creating the repository for the images

In [ ]:
for index, url in enumerate(tqdm(listeDesUrlDesImages)): # Here we write all the images contained in our listeDesUrlImages
    imageData = requests.get(url).content
    with open(f'/content/imagesTéléchargés/{index}.jpg', 'wb') as image_file:
        image_file.write(imageData)
    time.sleep(2) # We include a 2-second pause to avoid overloading the server.

# Example of IIIF usage: Retrieving annotations via Aikon

This example serves as a demonstration of the utility of being able to automatically retrieve large volumes of images and associated metadata using IIIF.

It is based on this [CLI application](https://github.com/Anarchiviste/aikoncli/blob/main/aikoncli.py).

In [ ]:
from pathlib import Path

url_manifest = "https://vhs.huma-num.fr/vhs/iiif/wit2320_img2562/manifest.json"
output_dir = "IIIFToYolo"

output_path = Path(output_dir)
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
import requests
import re
import json

from pathlib import Path
from tqdm import tqdm

def iiif_url_to_annotation_url(url_manifest: str) -> str:
    """
    Converts a IIIF manifest URL to its corresponding annotation URL.

    :param url_manifest: The IIIF manifest URL.
    :return: The annotation URL derived from the manifest URL.
    :raises ValueError: If the URL does not match the expected pattern.
    """
    match = re.search(r"(^https:\/\/.*)\/(.*)\/(.*\/wit)(\d+)(.*)\/(manifest\.json)", url_manifest)

    if match:
        radical_url = match.group(1)
        instance = match.group(2)
        witness_id = match.group(4)
        url_annotations = f"{radical_url}/{instance}/witness/{witness_id}/regions/canvas"
    else:
        raise ValueError('Invalide link')

    return url_annotations

def downloading_image_from_manifest(url_manifest: str, output_dir: Path) -> None:
    """
    Downloads all images from a IIIF manifest and saves them locally.
    Also builds a dictionary mapping image IDs to their metadata (url, height, width).

    :param url_manifest: The IIIF manifest URL.
    :param output_dir: The directory where images will be saved.
    :return: A tuple containing the image metadata dictionary and the witness name.
    """
    request = requests.get(url_manifest)
    manifest = request.json()
    metadata = manifest['metadata']

    meta = {item['label']: item['value'] for item in metadata}
    name = meta['@id'].replace(' ', '')
    images_dir = Path(f"{output_dir}/{name}/images")
    images_dir.mkdir(parents=True, exist_ok=True)

    canvases = manifest['sequences'][0]['canvases']
    i_id = 0
    img_dict = {}
    for canvas in tqdm(canvases, desc="Downloading"):
        for image in canvas['images']:
            resource = image['resource']
            i_id += 1
            img_dict[i_id] = {"url": resource['@id'], "height": resource['height'], "width": resource['width']}
            data = requests.get(resource['@id']).content
            with open(f'{images_dir}/{i_id}.jpg', 'wb') as f:
                f.write(data)

    return img_dict, name

def iiif_annotations_to_yolo(img_dict: dict, url_annotations: str, output_dir: Path, name: str) -> None:
    """
    Fetches annotations from an AIKON API endpoint and converts them to YOLO format label files.
    Each annotation's bounding box is normalized relative to its image dimensions.

    :param img_dict: Dictionary mapping image IDs to their metadata (url, height, width).
    :param url_annotations: The URL of the annotation endpoint.
    :param output_dir: The directory where label files will be saved.
    :param name: The witness name, used to build the output directory path.
    """
    labels_dir = Path(f"{output_dir}/{name}/labels")
    labels_dir.mkdir(parents=True, exist_ok=True)
    resultat_annotations = requests.get(url_annotations)
    print("Status :", resultat_annotations.status_code)
    print("Contenu :", repr(resultat_annotations.text[:300]))
    annotations_manifest = resultat_annotations.json()

    i_index = []

    for i in annotations_manifest:
        annotations = annotations_manifest.get(i)
        page_dictionnaire = img_dict.get(int(i))
        img_height = page_dictionnaire.get('height')
        img_width = page_dictionnaire.get('width')
        lines = {}

        for annotation in annotations:
            img_id = (annotations.get(annotation).get('canvas'))
            x_tl, y_tl, w, h = annotations.get(annotation).get('xywh')
            x_center = (x_tl + w / 2) / img_width
            y_center = (y_tl + h / 2) / img_height
            width = w / img_width
            height = h / img_height

            lines.setdefault(img_id, []).append(f'0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}')

        for img_id, img_lines in lines.items():
            with open(f'{labels_dir}/{img_id}.txt', 'w') as f:
                f.write("\n".join(img_lines))

def aikon_to_yolo(url_manifest: str, output_dir: str):
    """
    Orchestrates the full pipeline: converts a IIIF manifest URL to an annotation URL,
    downloads all images, and converts annotations to YOLO format label files.

    :param url_manifest: The IIIF manifest URL.
    :param output_dir: The directory where all outputs will be saved.
    """
    url_annotations = iiif_url_to_annotation_url(url_manifest)
    img_dict, name = downloading_image_from_manifest(url_manifest, output_dir)
    iiif_annotations_to_yolo(img_dict, url_annotations, output_dir, name)

aikon_to_yolo(url_manifest, output_dir)